In [1]:

import torch
import transformers
import datasets
import nltk
import networkx
import sklearn
import pandas as pd

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
Transformers: 5.22.1
Datasets: 4.8.5
GPU Available: True
GPU: Tesla T4


In [15]:
import time

print("Time module imported successfully!")

Time module imported successfully!


In [3]:
pip install -q transformers datasets torch nltk networkx scikit-learn rouge-score pandas matplotlib tqdm accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import re
import os
import numpy as np
import pandas as pd
import nltk
import networkx as nx
import torch

from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset
from transformers import pipeline
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

nltk.download("punkt")
nltk.download("punkt_tab")

print("All libraries imported successfully!")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


All libraries imported successfully!


In [5]:
dataset = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0",
    split={
        "train": "train[:500]",
        "validation": "validation[:100]",
        "test": "test[:100]"
    }
)

train_data = dataset["train"]
validation_data = dataset["validation"]
test_data = dataset["test"]

print("Training records:", len(train_data))
print("Validation records:", len(validation_data))
print("Testing records:", len(test_data))

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  259MB            

... (5 outputs left out)

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Training records: 500
Validation records: 100
Testing records: 100


In [6]:
sample = train_data[0]

print("ARTICLE:")
print(sample["article"][:1000])

print("\nREFERENCE SUMMARY:")
print(sample["highlights"])

ARTICLE:
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office chart. Detai

In [7]:
for split_name, split_data in dataset.items():
    empty_articles = sum(
        1 for x in split_data
        if not x["article"].strip()
    )

    empty_summaries = sum(
        1 for x in split_data
        if not x["highlights"].strip()
    )

    print(
        split_name,
        "Empty articles:", empty_articles,
        "Empty summaries:", empty_summaries
    )


train Empty articles: 0 Empty summaries: 0
validation Empty articles: 0 Empty summaries: 0
test Empty articles: 0 Empty summaries: 0


In [8]:
def clean_text(text):
    text = str(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def preprocess_dataset(data):
    records = []

    for item in data:
        article = clean_text(item["article"])
        summary = clean_text(item["highlights"])

        if article and summary:
            records.append({
                "article": article,
                "reference": summary
            })

    return pd.DataFrame(records)


train_df = preprocess_dataset(train_data)
val_df = preprocess_dataset(validation_data)
test_df = preprocess_dataset(test_data)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

Training: (500, 2)
Validation: (100, 2)
Testing: (100, 2)


In [9]:
def textrank_summarizer(text, num_sentences=3):

    sentences = sent_tokenize(text)

    if len(sentences) <= num_sentences:
        return " ".join(sentences)

    # Convert sentences into TF-IDF vectors
    vectorizer = TfidfVectorizer(stop_words="english")

    try:
        sentence_vectors = vectorizer.fit_transform(sentences)
    except ValueError:
        return " ".join(sentences[:num_sentences])

    # Calculate sentence similarity
    similarity_matrix = (
        sentence_vectors * sentence_vectors.T
    ).toarray()

    # Remove self-similarity
    np.fill_diagonal(similarity_matrix, 0)

    # Create sentence graph
    graph = nx.from_numpy_array(similarity_matrix)

    # Apply PageRank
    scores = nx.pagerank(graph, max_iter=100)

    # Rank sentences
    ranked_sentences = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    # Select important sentences
    selected_indices = sorted(
        ranked_sentences[:num_sentences]
    )

    # Preserve original sentence order
    summary = " ".join(
        sentences[i] for i in selected_indices
    )

    return summary


print("TextRank function created successfully!")

TextRank function created successfully!


In [10]:

sample_article = test_df.iloc[0]["article"]

textrank_summary = textrank_summarizer(
    sample_article,
    num_sentences=3
)

print("ORIGINAL ARTICLE:")
print(sample_article[:1500])

print("\nTEXTRANK SUMMARY:")
print(textrank_summary)

ORIGINAL ARTICLE:
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednes

In [16]:

 scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def calculate_rouge(reference, generated):

    scores = scorer.score(
        str(reference),
        str(generated)
    )

    return {
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure
    }

In [17]:
baseline_results = []

for index, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Evaluating TextRank"
):

    article = row["article"]
    reference = row["reference"]

    start_time = time.time()

    generated_summary = textrank_summarizer(
        article,
        num_sentences=3
    )

    execution_time = time.time() - start_time

    scores = calculate_rouge(
        reference,
        generated_summary
    )

    baseline_results.append({

        "Article_ID": index,

        "Reference": reference,

        "TextRank_Summary": generated_summary,

        "ROUGE1": scores["ROUGE-1"],

        "ROUGE2": scores["ROUGE-2"],

        "ROUGEL": scores["ROUGE-L"],

        "Execution_Time": execution_time
    })


baseline_results_df = pd.DataFrame(baseline_results)

print("Baseline evaluation completed!")

display(baseline_results_df.head())

Evaluating TextRank:   0%|          | 0/100 [00:00<?, ?it/s]

Baseline evaluation completed!


,Article_ID,Reference,TextRank_Summary,ROUGE1,ROUGE2,ROUGEL,Execution_Time
0,0,Membership gives the ICC jurisdiction over all...,(CNN)The Palestinian Authority officially beca...,0.238806,0.106061,0.194030,0.018354
1,1,"Theia, a bully breed mix, was apparently hit b...",That's according to Washington State Universit...,0.232558,0.031496,0.139535,0.007843
2,2,Mohammad Javad Zarif has spent more time with ...,"The website of the Iranian Foreign Ministry, w...",0.322581,0.175824,0.258065,0.009687
3,3,17 Americans were exposed to the Ebola virus w...,(CNN)Five Americans who were monitored for thr...,0.525424,0.241379,0.355932,0.007065
4,4,Student is no longer on Duke University campus...,(CNN)A Duke student has admitted to hanging a ...,0.549020,0.160000,0.254902,0.007031


In [18]:
baseline_performance = pd.DataFrame({

    "Metric": [
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Execution Time"
    ],

    "TextRank": [

        baseline_results_df["ROUGE1"].mean(),

        baseline_results_df["ROUGE2"].mean(),

        baseline_results_df["ROUGEL"].mean(),

        baseline_results_df["Execution_Time"].mean()
    ]
})

print("BASELINE MODEL PERFORMANCE")

display(baseline_performance.round(4))

BASELINE MODEL PERFORMANCE


,Metric,TextRank
0,ROUGE-1,0.2933
1,ROUGE-2,0.1063
2,ROUGE-L,0.2018
3,Execution Time,0.0094


In [19]:
baseline_results_df.to_csv(
    "baseline_textrank_results.csv",
    index=False
)

baseline_performance.to_csv(
    "baseline_performance.csv",
    index=False
)

print("Baseline model results saved successfully!")

Baseline model results saved successfully!
